# 01 — MLflow Overview & Experiment Tracking

**UI tabs:** Experiments · Overview · Metrics · Artifacts

| MLflow concept | API |
|---|---|
| Group related runs | `mlflow.set_experiment()` |
| Log config values | `mlflow.log_param()` |
| Log measured values | `mlflow.log_metric()` |
| Label a run | `mlflow.set_tag()` |
| Save a file | `mlflow.log_artifact()` |

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-genai --quiet

In [ ]:
import os, time, tempfile
from google import genai
from google.genai import types
import mlflow

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("01-MLflow-Overview")

print("MLflow", mlflow.__version__, "ready")

## Run 1 — Log params, metrics and a tag

In [ ]:
with mlflow.start_run(run_name="first-run"):
    t0 = time.time()
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=["What is MLflow? Answer in 2 sentences."]
    )
    latency = round(time.time() - t0, 3)

    mlflow.log_param("model", "gemini-2.5-flash")
    mlflow.log_param("question", "What is MLflow?")
    mlflow.log_metric("latency_seconds", latency)
    mlflow.log_metric("word_count", len(response.text.split()))
    mlflow.set_tag("use_case", "demo")

    print(response.text)
    print(f"\nLatency: {latency}s | Words: {len(response.text.split())}")

Open `http://127.0.0.1:5000` → click **01-MLflow-Overview** → click **first-run** → see Parameters, Metrics, Tags.

## Run 2 — Compare 3 temperatures (multi-run comparison)

In [ ]:
for temp in [0.0, 0.5, 1.0]:
    with mlflow.start_run(run_name=f"temp-{temp}"):
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=["Explain MLflow in one sentence."],
            config=types.GenerateContentConfig(temperature=temp, max_output_tokens=100)
        )

        mlflow.log_param("temperature", temp)
        mlflow.log_param("model", "gemini-2.5-flash")
        mlflow.log_metric("word_count", len(resp.text.split()))

        print(f"[temp={temp}] {resp.text[:120]}")

In the MLflow UI: select all 3 temperature runs → click **Compare** to see a side-by-side chart.

## Run 3 — Log step-based metrics (shows as a chart)

In [ ]:
questions = [
    "What is MLflow?",
    "What is experiment tracking?",
    "What is a model registry?"
]

with mlflow.start_run(run_name="step-metrics"):
    for step, q in enumerate(questions):
        t0 = time.time()
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[q]
        )
        lat = round(time.time() - t0, 3)

        # step= creates a chart in the MLflow UI (x-axis = step number)
        mlflow.log_metrics({"latency": lat, "words": len(resp.text.split())}, step=step)
        print(f"Step {step}: latency={lat}s words={len(resp.text.split())}")

Open the **step-metrics** run → click any metric name → see a line chart over steps.

## Run 4 — Log a text file as an artifact

In [ ]:
with mlflow.start_run(run_name="artifact-demo"):
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=["List 3 benefits of MLflow."]
    )

    # Save the response as a text file, then log it
    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
        f.write(resp.text)
        tmp_path = f.name

    mlflow.log_artifact(tmp_path, artifact_path="responses")
    mlflow.log_metric("word_count", len(resp.text.split()))

    print(resp.text)
    print("\nCheck the Artifacts tab in this run to see the saved file.")

## MLflow UI — What to explore
```
01-MLflow-Overview experiment
├── first-run          → Overview: params, metrics, tags
├── temp-0.0 / 0.5 / 1.0  → select all 3 → Compare
├── step-metrics       → click a metric → line chart
└── artifact-demo      → Artifacts tab → responses/
```
**Next →** `02_mlflow_tracing_gemini.ipynb`